# 🏗️ Detección de Equipo Pesado y Maquinaria en Sitios de Construcción
## Visión Artificial — Maestría en IA para AECO

**Objetivo:** Entrenar un modelo YOLOv8n para detección de equipo de seguridad, maquinaria y vehículos en obras de construcción.

**Dataset:** [Construction Site Safety](https://universe.roboflow.com/roboflow-universe-projects/construction-site-safety) — Roboflow Universe (CC BY 4.0)

**Pipeline:**
1. Configuración del entorno
2. Descarga del dataset
3. Creación de subconjunto (~200 train / ~50 valid)
4. Exploración visual del dataset
4b. Ejemplos de anotación (Ground Truth con bboxes)
4c. Inferencia baseline (modelo COCO sin fine-tuning)
5. Entrenamiento YOLOv8n (30 épocas)
6. Visualización de curvas de entrenamiento
7. Evaluación con métricas por clase
8. Inferencia en imágenes de validación
9. Inferencia en imágenes nuevas
10. Exploración con SAM (segmentación zero-shot)
11. Resumen y empaquetado de evidencias

---
## 1. Configuración del Entorno

In [ ]:
# Verificar GPU disponible
!nvidia-smi

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Instalar dependencias
!pip install ultralytics roboflow matplotlib seaborn pandas -q

In [ ]:
import os
import random
import shutil
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import pandas as pd
from pathlib import Path
from collections import Counter

# Semilla para reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("✅ Dependencias importadas correctamente")

---
## 2. Descarga del Dataset desde Roboflow

In [ ]:
from roboflow import Roboflow

# ⚠️ Reemplaza con tu API key de Roboflow (https://app.roboflow.com/settings/api)
rf = Roboflow(api_key="TU_API_KEY_AQUI")
project = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
version = project.version(28)  # Versión más reciente con YOLOv8
dataset = version.download("yolov8", location="/content/dataset_full")

print(f"\n✅ Dataset descargado en: {dataset.location}")

In [ ]:
# Explorar estructura del dataset descargado
dataset_root = "/content/dataset_full"

for split in ["train", "valid", "test"]:
    img_dir = os.path.join(dataset_root, split, "images")
    lbl_dir = os.path.join(dataset_root, split, "labels")
    if os.path.exists(img_dir):
        n_imgs = len(os.listdir(img_dir))
        n_lbls = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
        print(f"{split}: {n_imgs} imágenes, {n_lbls} etiquetas")
    else:
        print(f"{split}: No encontrado")

# Leer data.yaml para obtener nombres de clases
import yaml
yaml_path = os.path.join(dataset_root, "data.yaml")
with open(yaml_path, 'r') as f:
    data_yaml = yaml.safe_load(f)

class_names = data_yaml['names']
print(f"\nClases ({len(class_names)}): {class_names}")

---
## 3. Creación de Subconjunto (~200 train / ~50 valid)

Según las instrucciones del profesor, usaremos un subconjunto reducido del dataset.

In [ ]:
def create_subset(src_root, dst_root, split, n_samples, seed=42):
    """Crea un subconjunto aleatorio de imágenes y etiquetas."""
    random.seed(seed)

    src_imgs = os.path.join(src_root, split, "images")
    src_lbls = os.path.join(src_root, split, "labels")
    dst_imgs = os.path.join(dst_root, split, "images")
    dst_lbls = os.path.join(dst_root, split, "labels")

    os.makedirs(dst_imgs, exist_ok=True)
    os.makedirs(dst_lbls, exist_ok=True)

    # Listar imágenes disponibles
    all_images = sorted([f for f in os.listdir(src_imgs)
                         if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

    # Seleccionar subconjunto aleatorio
    n_samples = min(n_samples, len(all_images))
    selected = random.sample(all_images, n_samples)

    copied = 0
    for img_name in selected:
        # Copiar imagen
        shutil.copy2(os.path.join(src_imgs, img_name), os.path.join(dst_imgs, img_name))

        # Copiar etiqueta correspondiente
        lbl_name = os.path.splitext(img_name)[0] + ".txt"
        lbl_src = os.path.join(src_lbls, lbl_name)
        if os.path.exists(lbl_src):
            shutil.copy2(lbl_src, os.path.join(dst_lbls, lbl_name))
        copied += 1

    print(f"  {split}: {copied} imágenes copiadas de {len(all_images)} disponibles")
    return selected

# Crear subconjunto
subset_root = "/content/dataset_subset"
if os.path.exists(subset_root):
    shutil.rmtree(subset_root)

print("Creando subconjunto del dataset...")
train_selected = create_subset(dataset_root, subset_root, "train", n_samples=200, seed=SEED)
valid_selected = create_subset(dataset_root, subset_root, "valid", n_samples=50, seed=SEED)

# Guardar lista de archivos seleccionados para reproducibilidad
with open(os.path.join(subset_root, "train_files.txt"), "w") as f:
    f.write("\n".join(sorted(train_selected)))
with open(os.path.join(subset_root, "valid_files.txt"), "w") as f:
    f.write("\n".join(sorted(valid_selected)))

print("\n✅ Subconjunto creado exitosamente")
print(f"   Archivos de trazabilidad guardados en {subset_root}")

In [ ]:
# Crear data.yaml para el subconjunto
subset_yaml = {
    'path': subset_root,
    'train': 'train/images',
    'val': 'valid/images',
    'names': data_yaml['names'],
    'nc': len(data_yaml['names'])
}

subset_yaml_path = os.path.join(subset_root, "data.yaml")
with open(subset_yaml_path, 'w') as f:
    yaml.dump(subset_yaml, f, default_flow_style=False)

print(f"✅ data.yaml del subconjunto creado: {subset_yaml_path}")
print(f"\nContenido:")
with open(subset_yaml_path, 'r') as f:
    print(f.read())

---
## 4. Exploración Visual del Dataset

In [ ]:
# Mostrar muestras de imágenes del subconjunto
train_imgs_dir = os.path.join(subset_root, "train", "images")
sample_imgs = sorted(os.listdir(train_imgs_dir))[:9]

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
fig.suptitle("Muestras del Subconjunto de Entrenamiento", fontsize=16, fontweight='bold')

for idx, (ax, img_name) in enumerate(zip(axes.flatten(), sample_imgs)):
    img = mpimg.imread(os.path.join(train_imgs_dir, img_name))
    ax.imshow(img)
    ax.set_title(img_name[:30], fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig("/content/muestras_dataset.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Muestras guardadas en /content/muestras_dataset.png")

In [ ]:
# Distribución de clases en el subconjunto de entrenamiento
train_lbls_dir = os.path.join(subset_root, "train", "labels")
class_counts = Counter()

for lbl_file in os.listdir(train_lbls_dir):
    if lbl_file.endswith(".txt"):
        with open(os.path.join(train_lbls_dir, lbl_file), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cls_id = int(parts[0])
                    class_counts[cls_id] += 1

# Crear gráfico de barras
fig, ax = plt.subplots(figsize=(12, 6))
classes = [class_names[i] if i < len(class_names) else f"cls_{i}" for i in sorted(class_counts.keys())]
counts = [class_counts[i] for i in sorted(class_counts.keys())]

colors = sns.color_palette("husl", len(classes))
bars = ax.bar(classes, counts, color=colors, edgecolor='black', linewidth=0.5)

ax.set_xlabel("Clase", fontsize=12)
ax.set_ylabel("Número de Instancias", fontsize=12)
ax.set_title("Distribución de Clases — Subconjunto de Entrenamiento (~200 imgs)", fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
            str(count), ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig("/content/distribucion_clases.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nResumen de distribución:")
for cls, cnt in zip(classes, counts):
    print(f"  {cls}: {cnt} instancias")

---
## 4b. Ejemplos de Anotación (Ground Truth)

Visualizamos 5 imágenes del subconjunto con sus bounding boxes ground truth dibujados.

In [ ]:
import cv2

def draw_gt_boxes(img_path, lbl_path, class_names, colors_map=None):
    """Dibuja bounding boxes ground truth sobre una imagen."""
    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    if colors_map is None:
        np.random.seed(0)
        colors_map = {i: tuple(int(c) for c in np.random.randint(50, 255, 3)) for i in range(len(class_names))}

    if os.path.exists(lbl_path):
        with open(lbl_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                    x1 = int((cx - bw/2) * w)
                    y1 = int((cy - bh/2) * h)
                    x2 = int((cx + bw/2) * w)
                    y2 = int((cy + bh/2) * h)
                    color = colors_map.get(cls_id, (0, 255, 0))
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                    label = class_names[cls_id] if cls_id < len(class_names) else f'cls_{cls_id}'
                    cv2.putText(img, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Seleccionar 5 imágenes con anotaciones
train_imgs_list = sorted(os.listdir(train_imgs_dir))[:5]

fig, axes = plt.subplots(1, 5, figsize=(25, 5))
fig.suptitle('Ejemplos de Anotación — Ground Truth Bounding Boxes', fontsize=16, fontweight='bold')

for idx, (ax, img_name) in enumerate(zip(axes.flatten(), train_imgs_list)):
    img_path = os.path.join(train_imgs_dir, img_name)
    lbl_name = os.path.splitext(img_name)[0] + '.txt'
    lbl_path = os.path.join(train_lbls_dir, lbl_name)
    annotated = draw_gt_boxes(img_path, lbl_path, class_names)
    ax.imshow(annotated)
    ax.set_title(img_name[:25], fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/ejemplos_anotacion_gt.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 5 ejemplos de anotación guardados en /content/ejemplos_anotacion_gt.png')

---
## 4c. Inferencia Baseline (Modelo Preentrenado COCO — Sin Fine-Tuning)

Antes de entrenar, evaluamos qué detecta el modelo YOLOv8n preentrenado en COCO en imágenes de construcción. Esto establece la **línea base** para medir la mejora tras el fine-tuning.

In [ ]:
from ultralytics import YOLO

# Cargar modelo preentrenado en COCO (80 clases genéricas)
baseline_model = YOLO('yolov8n.pt')

# Seleccionar 5 imágenes de validación para baseline
valid_imgs_baseline = sorted(os.listdir(os.path.join(subset_root, 'valid', 'images')))[:5]
baseline_paths = [os.path.join(subset_root, 'valid', 'images', f) for f in valid_imgs_baseline]

# Inferencia baseline
baseline_results = baseline_model.predict(
    source=baseline_paths,
    imgsz=640,
    conf=0.25,
    verbose=False
)

fig, axes = plt.subplots(1, 5, figsize=(25, 5))
fig.suptitle('Inferencia Baseline — YOLOv8n Preentrenado COCO (SIN fine-tuning)', fontsize=16, fontweight='bold')

for idx, (ax, result) in enumerate(zip(axes.flatten(), baseline_results)):
    img_plot = result.plot()
    ax.imshow(img_plot[..., ::-1])
    n_det = len(result.boxes)
    ax.set_title(f'{valid_imgs_baseline[idx][:20]}\n({n_det} dets)', fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/inferencia_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 Análisis Baseline:')
print('  El modelo COCO detecta personas y algunos vehículos genéricos,')
print('  pero NO detecta clases específicas de seguridad como:')
print('  Hardhat, NO-Hardhat, Safety Vest, Safety Cone, machinery.')
print('  → El fine-tuning es necesario para el dominio AECO.')

---
## 5. Entrenamiento YOLOv8n (30 Épocas)

In [ ]:
from ultralytics import YOLO

# Cargar modelo preentrenado YOLOv8n
model = YOLO("yolov8n.pt")

print("✅ Modelo YOLOv8n cargado")
print(f"   Arquitectura: YOLOv8 Nano")
print(f"   Parámetros: ~3.2M")

In [ ]:
# Entrenamiento
results = model.train(
    data=subset_yaml_path,
    epochs=30,
    batch=16,
    imgsz=640,
    seed=SEED,
    project="/content/yolo_training",
    name="construction_safety",
    exist_ok=True,
    verbose=True
)

print("\n✅ Entrenamiento completado")

---
## 6. Visualización de Curvas de Entrenamiento

In [ ]:
# Buscar directorio de resultados con glob recursivo
train_dir_candidates = glob.glob("/content/yolo_training/construction_safety*/", recursive=False)
if not train_dir_candidates:
    train_dir_candidates = glob.glob("/content/yolo_training/**/results.png", recursive=True)
    if train_dir_candidates:
        train_dir = os.path.dirname(train_dir_candidates[0])
    else:
        train_dir = "/content/yolo_training/construction_safety"
else:
    train_dir = train_dir_candidates[-1].rstrip('/')

print(f"Directorio de entrenamiento: {train_dir}")
print(f"\nArchivos disponibles:")
for f in sorted(os.listdir(train_dir)):
    print(f"  {f}")

In [ ]:
# Mostrar results.png (curvas de entrenamiento generadas por YOLO)
results_img = os.path.join(train_dir, "results.png")
if os.path.exists(results_img):
    fig, ax = plt.subplots(figsize=(18, 10))
    img = mpimg.imread(results_img)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title("Curvas de Entrenamiento YOLOv8n — 30 Épocas", fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ results.png no encontrado. Verificar ruta de entrenamiento.")

In [ ]:
# Mostrar matriz de confusión
confusion_paths = glob.glob(os.path.join(train_dir, "*confusion*"), recursive=False)
if confusion_paths:
    fig, axes = plt.subplots(1, len(confusion_paths), figsize=(10*len(confusion_paths), 10))
    if len(confusion_paths) == 1:
        axes = [axes]
    for ax, cpath in zip(axes, confusion_paths):
        img = mpimg.imread(cpath)
        ax.imshow(img)
        ax.set_title(os.path.basename(cpath), fontsize=12)
        ax.axis('off')
    plt.suptitle("Matrices de Confusión", fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No se encontraron matrices de confusión.")

# Nota: P_curve.png, R_curve.png, F1_curve.png, PR_curve.png son opcionales.
# Si existen, se muestran; si no, es normal.

---
## 7. Evaluación en Validación

In [ ]:
# Cargar el mejor modelo entrenado
best_model_path = os.path.join(train_dir, "weights", "best.pt")
if not os.path.exists(best_model_path):
    # Buscar recursivamente
    candidates = glob.glob("/content/yolo_training/**/best.pt", recursive=True)
    if candidates:
        best_model_path = candidates[0]

print(f"Modelo: {best_model_path}")
best_model = YOLO(best_model_path)

In [ ]:
# Evaluar en validación
val_results = best_model.val(
    data=subset_yaml_path,
    split='val',
    imgsz=640,
    batch=16,
    verbose=True
)

print("\n" + "="*70)
print("MÉTRICAS GLOBALES DE EVALUACIÓN")
print("="*70)
print(f"  Precision (P):     {val_results.results_dict['metrics/precision(B)']:.4f}")
print(f"  Recall (R):        {val_results.results_dict['metrics/recall(B)']:.4f}")
print(f"  mAP@0.5:           {val_results.results_dict['metrics/mAP50(B)']:.4f}")
print(f"  mAP@0.5:0.95:      {val_results.results_dict['metrics/mAP50-95(B)']:.4f}")
print("="*70)

In [ ]:
# Tabla de métricas por clase
print("\n" + "="*80)
print(f"{'Clase':<20} {'Precision':>10} {'Recall':>10} {'mAP@50':>10} {'mAP@50-95':>12}")
print("-"*80)

# Extraer métricas por clase del objeto de resultados
box_results = val_results.box
for i, cls_name in enumerate(class_names):
    if i < len(box_results.p):
        p = box_results.p[i]
        r = box_results.r[i]
        ap50 = box_results.ap50[i]
        ap = box_results.ap[i]
        print(f"{cls_name:<20} {p:>10.4f} {r:>10.4f} {ap50:>10.4f} {ap:>12.4f}")

print("-"*80)
print(f"{'PROMEDIO':<20} {box_results.mp:>10.4f} {box_results.mr:>10.4f} {box_results.map50:>10.4f} {box_results.map:>12.4f}")
print("="*80)

---
## 8. Inferencia en Imágenes de Validación (Grid Visual)

In [ ]:
# Seleccionar 10 imágenes de validación al azar
valid_imgs_dir = os.path.join(subset_root, "valid", "images")
valid_imgs = sorted([f for f in os.listdir(valid_imgs_dir)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

random.seed(SEED)
sample_valid = random.sample(valid_imgs, min(10, len(valid_imgs)))
sample_valid_paths = [os.path.join(valid_imgs_dir, f) for f in sample_valid]

# Ejecutar inferencia
pred_results = best_model.predict(
    source=sample_valid_paths,
    imgsz=640,
    conf=0.25,
    save=True,
    project="/content/predictions",
    name="valid_inference",
    exist_ok=True
)

print(f"✅ Inferencia completada en {len(sample_valid)} imágenes de validación")

In [ ]:
# Buscar carpeta real de predicciones con glob recursivo
pred_dirs = glob.glob("/content/predictions/**/", recursive=True)
pred_images = []
for pd_dir in pred_dirs:
    pred_images.extend(glob.glob(os.path.join(pd_dir, "*.jpg")))
    pred_images.extend(glob.glob(os.path.join(pd_dir, "*.jpeg")))
    pred_images.extend(glob.glob(os.path.join(pd_dir, "*.png")))

# Si no encontró imágenes, intentar desde plot de los resultados
if not pred_images:
    for r in pred_results:
        if hasattr(r, 'save_dir'):
            save_dir = str(r.save_dir)
            pred_images = glob.glob(os.path.join(save_dir, "*.*"))
            break

pred_images = sorted(pred_images)[:10]
print(f"Imágenes con predicción encontradas: {len(pred_images)}")

# Visualizar grid 2x5
if pred_images:
    n_cols = 5
    n_rows = 2
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(25, 10))
    fig.suptitle("Inferencia YOLOv8n — Imágenes de Validación", fontsize=18, fontweight='bold')

    for idx, ax in enumerate(axes.flatten()):
        if idx < len(pred_images):
            img = mpimg.imread(pred_images[idx])
            ax.imshow(img)
            ax.set_title(os.path.basename(pred_images[idx])[:25], fontsize=8)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig("/content/grid_validacion.png", dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("⚠️ No se encontraron imágenes con predicciones. Revisar rutas.")

---
## 9. Inferencia en 5 Imágenes Nuevas (del dataset original)

Usamos imágenes del dataset completo que **NO** están en nuestro subconjunto.

In [ ]:
# Encontrar imágenes del dataset original que NO estén en el subconjunto
all_train_imgs = set(os.listdir(os.path.join(dataset_root, "train", "images")))
all_valid_imgs = set(os.listdir(os.path.join(dataset_root, "valid", "images")))

# Imágenes ya usadas en subconjunto
used_imgs = set(train_selected + valid_selected)

# Buscar imágenes no usadas (primero en valid original, luego en train original)
available_new = []
for img_name in sorted(all_valid_imgs - used_imgs):
    if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
        available_new.append(os.path.join(dataset_root, "valid", "images", img_name))

for img_name in sorted(all_train_imgs - used_imgs):
    if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
        available_new.append(os.path.join(dataset_root, "train", "images", img_name))

# También buscar en test si existe
test_dir = os.path.join(dataset_root, "test", "images")
if os.path.exists(test_dir):
    for img_name in sorted(os.listdir(test_dir)):
        if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            available_new.append(os.path.join(test_dir, img_name))

random.seed(SEED + 1)
new_images = random.sample(available_new, min(5, len(available_new)))

print(f"Imágenes nuevas seleccionadas: {len(new_images)}")
for p in new_images:
    print(f"  {os.path.basename(p)}")

In [ ]:
# Inferencia en imágenes nuevas
new_results = best_model.predict(
    source=new_images,
    imgsz=640,
    conf=0.25,
    save=True,
    project="/content/predictions",
    name="new_inference",
    exist_ok=True
)

# Buscar imágenes guardadas con glob recursivo
new_pred_imgs = []
for pat in ["/content/predictions/new_inference*/*.jpg",
            "/content/predictions/new_inference*/*.jpeg",
            "/content/predictions/new_inference*/*.png"]:
    new_pred_imgs.extend(glob.glob(pat))

new_pred_imgs = sorted(new_pred_imgs)[:5]

# Visualizar
if new_pred_imgs:
    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    fig.suptitle("Inferencia en Imágenes Nuevas (fuera del subconjunto)", fontsize=16, fontweight='bold')

    for idx, ax in enumerate(axes.flatten()):
        if idx < len(new_pred_imgs):
            img = mpimg.imread(new_pred_imgs[idx])
            ax.imshow(img)
            ax.set_title(os.path.basename(new_pred_imgs[idx])[:25], fontsize=8)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig("/content/grid_nuevas.png", dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("⚠️ No se encontraron imágenes de predicción nuevas.")

---
## 10. Exploración con SAM — Segmentación Zero-Shot

In [ ]:
# Cargar MobileSAM
from ultralytics import SAM

sam_model = SAM("mobile_sam.pt")
print("✅ MobileSAM cargado correctamente")

In [ ]:
# Seleccionar 3 imágenes de validación para demostración SAM
sam_images = sample_valid_paths[:3]

fig, axes = plt.subplots(3, 3, figsize=(20, 20))
fig.suptitle("Comparación: Original → YOLOv8 Detección → SAM Segmentación", fontsize=18, fontweight='bold')

for row_idx, img_path in enumerate(sam_images):
    # Columna 1: Imagen original
    original = mpimg.imread(img_path)
    axes[row_idx, 0].imshow(original)
    axes[row_idx, 0].set_title("Original", fontsize=12)
    axes[row_idx, 0].axis('off')

    # Columna 2: Detección YOLOv8
    yolo_pred = best_model.predict(img_path, imgsz=640, conf=0.25, verbose=False)
    yolo_img = yolo_pred[0].plot()
    axes[row_idx, 1].imshow(yolo_img[..., ::-1])  # BGR -> RGB
    axes[row_idx, 1].set_title("YOLOv8 Detección", fontsize=12)
    axes[row_idx, 1].axis('off')

    # Columna 3: SAM segmentación usando bboxes de YOLO
    if len(yolo_pred[0].boxes) > 0:
        bboxes = yolo_pred[0].boxes.xyxy.cpu().numpy()
        sam_results = sam_model.predict(
            img_path,
            bboxes=bboxes,
            verbose=False
        )
        sam_img = sam_results[0].plot()
        axes[row_idx, 2].imshow(sam_img[..., ::-1])
        axes[row_idx, 2].set_title(f"SAM Segmentación ({len(bboxes)} objetos)", fontsize=12)
    else:
        axes[row_idx, 2].imshow(original)
        axes[row_idx, 2].set_title("SAM: Sin detecciones", fontsize=12)
    axes[row_idx, 2].axis('off')

plt.tight_layout()
plt.savefig("/content/comparacion_sam.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Comparación YOLO vs SAM completada")
print("   SAM genera máscaras de segmentación pixel a pixel")
print("   usando los bounding boxes de YOLO como prompts.")

---
## 11. Resumen Final y Empaquetado de Evidencias

In [ ]:
# Resumen de resultados
print("\n" + "="*70)
print("  RESUMEN DEL PROYECTO")
print("="*70)
print(f"\n  Modelo:          YOLOv8n (Nano)")
print(f"  Dataset:         Construction Site Safety (Roboflow)")
print(f"  Subconjunto:     ~200 train / ~50 valid")
print(f"  Épocas:          30")
print(f"  Batch size:      16")
print(f"  Imagen:          640x640")
print(f"  Seed:            {SEED}")
print(f"\n  --- Métricas Globales ---")
print(f"  Precision:       {val_results.results_dict['metrics/precision(B)']:.4f}")
print(f"  Recall:          {val_results.results_dict['metrics/recall(B)']:.4f}")
print(f"  mAP@0.5:         {val_results.results_dict['metrics/mAP50(B)']:.4f}")
print(f"  mAP@0.5:0.95:    {val_results.results_dict['metrics/mAP50-95(B)']:.4f}")
print("="*70)

In [ ]:
# Empaquetar evidencias en ZIP
import zipfile

zip_path = "/content/evidencias_proyecto.zip"

evidence_files = {
    "muestras_dataset.png": "/content/muestras_dataset.png",
    "distribucion_clases.png": "/content/distribucion_clases.png",
    "ejemplos_anotacion_gt.png": "/content/ejemplos_anotacion_gt.png",
    "inferencia_baseline.png": "/content/inferencia_baseline.png",
    "grid_validacion.png": "/content/grid_validacion.png",
    "grid_nuevas.png": "/content/grid_nuevas.png",
    "comparacion_sam.png": "/content/comparacion_sam.png",
}

# Agregar archivos del entrenamiento
training_files = glob.glob(os.path.join(train_dir, "*.png")) + \
                 glob.glob(os.path.join(train_dir, "*.csv"))

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Evidencias principales
    for name, path in evidence_files.items():
        if os.path.exists(path):
            zf.write(path, f"evidencias/{name}")

    # Archivos de entrenamiento YOLO
    for f in training_files:
        zf.write(f, f"entrenamiento/{os.path.basename(f)}")

    # Mejor modelo
    if os.path.exists(best_model_path):
        zf.write(best_model_path, "modelo/best.pt")

    # Listas de archivos del subconjunto
    for fname in ["train_files.txt", "valid_files.txt"]:
        fpath = os.path.join(subset_root, fname)
        if os.path.exists(fpath):
            zf.write(fpath, f"subconjunto/{fname}")

print(f"✅ Evidencias empaquetadas en: {zip_path}")
print(f"   Tamaño: {os.path.getsize(zip_path) / 1024 / 1024:.1f} MB")

# Descargar desde Colab
from google.colab import files
files.download(zip_path)

---
## 12. Tabla de Prueba de Reproducibilidad

| Parámetro | Valor |
|---|---|
| **Dataset** | Construction Site Safety (Roboflow Universe) |
| **Versión Dataset** | v28 (YOLOv8) |
| **Subconjunto Train** | ~200 imágenes |
| **Subconjunto Valid** | ~50 imágenes |
| **Seed** | 42 |
| **Modelo** | YOLOv8n (yolov8n.pt) |
| **Épocas** | 30 |
| **Batch Size** | 16 |
| **Tamaño de Imagen** | 640×640 |
| **Framework** | Ultralytics (YOLO) |
| **SAM** | MobileSAM (mobile_sam.pt) |
| **GPU** | Google Colab T4 |
| **Python** | 3.10+ |
| **Licencia Dataset** | CC BY 4.0 |
| **Licencia Código** | MIT |
| **Licencia YOLOv8** | AGPL-3.0 |

---
### Versiones de Librerías

In [ ]:
import sys
import ultralytics
import roboflow

print("Versiones del entorno:")
print(f"  Python:       {sys.version.split()[0]}")
print(f"  PyTorch:      {torch.__version__}")
print(f"  CUDA:         {torch.version.cuda}")
print(f"  Ultralytics:  {ultralytics.__version__}")
print(f"  Roboflow:     {roboflow.__version__}")
print(f"  NumPy:        {np.__version__}")
print(f"  Matplotlib:   {plt.matplotlib.__version__}")
print(f"  Pandas:       {pd.__version__}")

print("\n🎉 ¡Pipeline completo! Todos los pasos ejecutados exitosamente.")